# Objective

This notebook should follow "Detect Bragg peaks.ipynb". Now that we've got the peaks for each diffraction pattern, we can calculate Datapoints to check for the presence of particular crystal phases at various locations in the nanowire. In this notebook, we'll do the following:

- Define "target" Datapoints for wurtzite (WZ), zincblende (ZB) and lead (Pb).
- Use these target Datapoints to generate a phase map showing the presence of the above 3 phases throughout the nanowire.
- Superimpose the phase maps to generate one descriptive image of the nanowire.

# Imports & data loading

In [ ]:
import py4DSTEM
import numpy as np
import matplotlib.pyplot as plt
from py4DSTEM import show
from scipy.spatial import KDTree
from scipy.ndimage import correlate
from scipy.optimize import curve_fit
import copy
from itertools import product

py4DSTEM.__version__

In [ ]:
plt.rcParams.update({
    'axes.titlesize': 8,        # Title of the axes
    'axes.labelsize': 8,        # Axis labels
    'xtick.labelsize': 8,       # X-axis tick labels
    'ytick.labelsize': 8,       # Y-axis tick labels
    'legend.fontsize': 8,       # Legend text
    'figure.titlesize': 8,      # Overall figure title (if using suptitle)
    'figure.dpi' : 100
})

In [ ]:
# Load data
data_path = 'file.h5'
datacube     = py4DSTEM.read(data_path)
pattern_data = datacube.data + 0

detected_peaks_arr = np.load('detected_peaks_arr.npy', allow_pickle=True)
alg_1_peaks_arr    = np.load('alg_1_peaks_arr.npy'   , allow_pickle=True)
vacuum_peaks_arr   = np.load('vacuum_peaks_arr.npy'  , allow_pickle=True)
alg_2_peaks_arr    = np.load('alg_2_peaks_arr.npy'   , allow_pickle=True)


In [ ]:
# Set important hyperparams.
eps_ratio  = 0.12
eps_angle  = 6
eps_length = 2

# Function definitions: crystal lattice identification

The functions defined in this section calculate Datapoints for a given diffraction pattern (`calculate_datapoints`), and check if these match with the provided "target" Datapoints from a crystal phase we are interested in (`calculate_datapoints`).

**TODO: All of this should go into a separate file for import.**

In [ ]:
def calculate_datapoints_helper(v1, v2, pair_id):
    """
    INPUT
    v1 , v2: The two (centered) Bragg vectors of form Array(qx, qy) to be compared.
    pair_id: Tuple of integers (i1, i2), which identify the Bragg vectors.
    OUTPUT
    datapoint: Structured array with dtype ( 'pair_id':.., 'ratio': .., 'angle' (degrees): .., 'length':..).
    'ratio' is the ratio of the two Bragg vector lengths. 'length' is the smaller of the two. 
    'angle_deg' is the angle between the vectors (in degrees).
    """
    
    # Find the ratio.
    l1 = np.linalg.norm(v1)
    l2 = np.linalg.norm(v2)
    if l1 >= l2:
        ratio = l1/l2
        minlength = l2
    else:
        ratio = l2/l1
        minlength = l1
    
    # Find the angle.
    cos_theta = np.clip( np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)), -1.0, 1.0 )
    angle_rad = np.arccos(cos_theta)
    angle_deg = angle_rad * 360 / (2*np.pi)
    
    # Make the Datapoint object.
    dtype     = np.dtype([ ('pair_ID', 'int16', (2,)), ('ratio', 'float64'),
                          ('angle', 'float64'), ('length', 'float64') ])
    datapoint = np.zeros(1, dtype=dtype)
    datapoint['pair_ID'] = pair_id
    datapoint['ratio']   = ratio
    datapoint['angle']   = angle_deg
    datapoint['length']  = minlength

    return datapoint

from itertools import combinations

def calculate_datapoints(chosen_peak_vectors_input):
    """
    INPUTS
    chosen_peak_vectors_input: Tuple (qx_list, qy_list) of centered peak positions.
    
    TODO: add explanation.
    """
    # First, rearrange tuple of peak positions as an N x 2 array. 1st column is x, 2nd is y.
    chosen_peak_vectors = np.column_stack(chosen_peak_vectors_input)
    
    # Define structured datatype and redefine the chosen Bragg peaks in this way.
    dtype       = np.dtype([('ID', 'int16'), ('value', 'float64', (2,))])
    bragg_peaks = np.zeros(len(chosen_peak_vectors), dtype=dtype)
    
    # Apply an ID to each Bragg peak.
    for id_num, peak in enumerate(chosen_peak_vectors):
        bragg_peaks['ID'][id_num]    = id_num
        bragg_peaks['value'][id_num] = peak # NOTE: It shouldn't matter which is x or y, if we are consistent.
        
    # Now that the chosen peaks are in structured form, take combinations and calculate datapoints.
    dtype     = np.dtype([ ('pair_ID', 'int16', (2,)), ('ratio', 'float64'),
                          ('angle', 'float64'), ('length', 'float64') ])
    datapoints = np.zeros(0, dtype=dtype)
    for pair in combinations(bragg_peaks, 2):
        id1, val1 = pair[0]['ID'], pair[0]['value']
        id2, val2 = pair[1]['ID'], pair[1]['value']
        datapoint = calculate_datapoints_helper(val1, val2, (id1, id2))
        datapoints = np.append(datapoints, datapoint)
        
    return datapoints, bragg_peaks

#################################################

def pair_checker(sample_datapoints, target_datapoints):
    """Each 'Datapoints' array has 3 entries. The function attempts to check if the 'sample' and 'target' datapoints
    are indeed a proper match. Returns True if yes, False otherwise.
    
    !! NOTE !!: It is expected that sample_datapoints[i] and target_datapoints[i] are within eps_X of each other, i.e.
    'sample_datapoints' was constructed in a special order w.r.t. 'target_datapoints'. Otherwise, this function breaks."""
    
    print(f"pair_checker(\n\033[94m{sample_datapoints}\033[0m, \n\033[92m{target_datapoints}\033[0m)\n")
    
    # Start by checking that only 3 Bragg vectors participate in the 'sample' datapoints.
    sample_IDs  = np.unique(sample_datapoints['pair_ID'])
    target_IDs = np.unique(target_datapoints['pair_ID'])
    if len(sample_IDs) != 3:
        return False, None, None
    
    # Rows i of the two Datapoints arrays correspond to each other. Use the first row to establish
    # correspondence of IDs (in a dictionary).
    row_0_IDs_sample  = sample_datapoints[0]['pair_ID']
    row_0_IDs_target = target_datapoints[0]['pair_ID']
    
    # Create the 'correspondence' dictionaries.
    correspondence_1 = {i : j for (i, j) in zip(row_0_IDs_target, row_0_IDs_sample)}
    correspondence_2 = {i : j for (i, j) in zip(row_0_IDs_target[::-1], row_0_IDs_sample)}
    
    # The last IDs of each Datapoints array must also correspond.
    last_ID_sample  = np.setdiff1d(sample_IDs , row_0_IDs_sample , assume_unique=True)[0]
    last_ID_target = np.setdiff1d(target_IDs, row_0_IDs_target, assume_unique=True)[0]
    
    correspondence_1.update({last_ID_target : last_ID_sample})
    correspondence_2.update({last_ID_target : last_ID_sample})
#     print(f"Correspondence_1 dictionary: {correspondence_1}")
#     print(f"Correspondence_2 dictionary: {correspondence_2}")
    
    # We need to see if either of the two correspondence dictionaries correctly predict a match
    # in the remaining 2 rows of Datapoints.
    corr_1_match_list, corr_2_match_list = [], []
    for row_sample, row_target in zip(sample_datapoints[1:], target_datapoints[1:]):
        sample_IDs = row_sample['pair_ID']
        target_IDs = row_target['pair_ID']
        
        # Check matches w.r.t. both dictionaries.
        corr_1_match_list.append(check_row_match(sample_IDs, target_IDs, correspondence_1))
        corr_2_match_list.append(check_row_match(sample_IDs, target_IDs, correspondence_2))
        
    # Check which (if either) of the dictionaries successfully matched. Return accordingly.
    corr_1_match = all(corr_1_match_list)
    corr_2_match = all(corr_2_match_list)
    match_outcome = corr_1_match or corr_2_match
    
    if match_outcome:
        correspondence = correspondence_1 if corr_1_match else correspondence_2
        matched_sample_vector_IDs = list(correspondence.values())
        matched_target_vector_IDs = list(correspondence.keys())
        
        """Besides the match verdict, we also need to return the 3 vector IDs. They need to be in an order
        respecting the 'target' datapoints, so that the parent function (check_match) knows which 'sample' and
        'target' vectors correspond."""
        return True, matched_sample_vector_IDs, matched_target_vector_IDs
    else:
        return False, None, None
    
def check_row_match(sample_IDs, target_IDs, correspondence):
    expected_sample_IDs = [correspondence[i] for i in target_IDs]
    match = (all(i in sample_IDs for i in expected_sample_IDs) and
             all(i in expected_sample_IDs for i in sample_IDs))
    return match

def check_datapoints_validity(three_datapoints):
    """Returns `True` if the provided Datapoints meet certain criteria, `False` otherwise."""
    # Start by checking that there are only 3 vectors in total.
    unique_vector_IDs, unique_ID_counts  = np.unique(three_datapoints['pair_ID'], return_counts=True)
    n = len(unique_vector_IDs)
    if n < 3:
        print(f"Error! There are only {n} vector IDs, we expected 3 or more! Throwing error.")
        print(1/0) # Can comment out; code runs, but needs to be looked at if this error occurs.
        return False
    elif n > 3:
        # This can happen, but makes the triplet invalid.
        return False
    
    # Check that the same vector doesn't appears twice in a given Datapoint. Should never happen if code
    # works correctly.
    if any(three_datapoints['pair_ID'][:, 0] == three_datapoints['pair_ID'][:, 1]):
        print("Error! There's a Datapoint constructed from a single vector. This is catastrophe.")
        print(1/0) # Do not comment this out. The logic error must be fixed.
        return False

    # Finally, check that each unique vector appears exactly twice in the three Datapoints.
    if all( unique_ID_counts == np.array([2, 2, 2]) ):
        return True
    else:
        False

#################################################

def check_match(sample_datapoints, sample_bragg_peaks, target_datapoints, target_bragg_peaks,
                eps_ratio, eps_angle, eps_length, use_almost = False):
    dtype = target_datapoints.dtype
    
    # Initialize variables to store match state for (each) label datapoint.
    match_list        = []
    ALMOST_match_list = []
    
    # Initialize a dictionary which contains the 'data' and 'label' datapoints which match.
    dict_of_matches = {}    
    
    # Iterate over the 'label' datapoints.
    for useless_counter, target_datapoint in enumerate(target_datapoints):

        dp_fits        = False # Tracks if DP matches current label datapoint.
        dp_ALMOST_fits = False # Tracks if DP matches current label datapoint, save for length.
        datapoints_that_match = [] # Tracks the 'data' datapoints matching the current 'label' datapoint.

        # Iterate over the 'data' DP's (ratio, angle, length) datapoints.
        for datapoint in sample_datapoints:

            # First, check if the eps_... thresholds are reasonable.
            if eps_ratio >= datapoint['ratio'] or eps_length >= datapoint['length']:# or eps_angle >= datapoint['angle']:
                print("Error! Tolerances are larger than actual values.")
                print(1/1)

            # Next, check if the current 'data' and 'label' datapoints match.
            if (np.abs(datapoint['ratio'] - target_datapoint['ratio']) < eps_ratio
                and np.abs(datapoint['angle'] - target_datapoint['angle']) < eps_angle):
                dp_ALMOST_fits = True
                if use_almost:
                    datapoints_that_match.append(datapoint)

                if np.abs(datapoint['length'] - target_datapoint['length']) < eps_length:
                    dp_fits = True
                    
                    """Here, we should append 'datapoint' to a list specifically designated to
                    the present 'target_datapoint'. We can make a dictionary where the 'target_datapoint'
                    is the key and this list is the value, and re-empty list for each new 'target_datapoint'.
                    Then we can easily make the combinations of 'data' datapoints and feed everything
                    into 'pair_checker'."""
                    # Append the current 'data' datapoint to the list of matches.
                    if not use_almost:
                        datapoints_that_match.append(datapoint)
                    
        # Now that we've iterated all the 'data' datapoints, we can add the 'label' datapoints and those
        # 'data' ones which match it as a key-value pair into the main dictionary.
        # TODO: This part must go. There's actually no reason to have a dictionary. List is ok.
        dict_of_matches.update({useless_counter : datapoints_that_match})

        # Append the current 'label' datapoint's match states to lists.
        match_list.append(dp_fits)
        ALMOST_match_list.append(dp_ALMOST_fits)

    # If every 'label' datapoint had a match, we say the two DPs' crystal structures match.
    match_verdict        = False not in match_list
    ALMOST_match_verdict = False not in ALMOST_match_list
    
    ### Choose which verdict to follow, depending on 'use_almost'.
    main_verdict = ALMOST_match_verdict if use_almost else match_verdict
    
    # If 'match_verdict' is True, then there is at least one 'data' datapoint matching each 'label' point.
    # We now have to check if 3 Bragg/lattice vectors are responsible for the matching, and extract them.
    # ! NOTE !: Due to how we built the dictionary, each entry in 'ALL_datpoints_that_match' corresponds
    # to each entry in 'target_datapoints'.
    if main_verdict:
        ALL_datapoints_that_match  = list(dict_of_matches.values()) # [dtpt_list_1, dtpt_list_2, dtpt_list_3]
        cartesian_product_iterator = product(*ALL_datapoints_that_match)
        
        for cartesian_product in cartesian_product_iterator:
            # 'cartesian_product' is a list of Datapoint elements. Make it a structured array.
            cart_prod_arr = np.array(list(cartesian_product), dtype=dtype)
            TRUE_match_verdict, sample_vector_IDs, target_vector_IDs = pair_checker(cart_prod_arr, target_datapoints)
            
            """If we truly have a match, we want to a) return the verdict and b) return Bragg vectors. In principle,
            we could extract the 'label' vectors separately, but I'll return them here also. We want the 'data' and
            'label' vectors to be match-able onto another, so that I can plot them together to visually see if they
            align as expected."""
            if TRUE_match_verdict:
#                 print("Debug print.")
#                 print(f"data peaks: {sample_bragg_peaks}")
#                 print(f"data peak IDs: {sample_bragg_peaks['ID']}")
#                 print(f"label peak IDs: {target_bragg_peaks['ID']}")
#                 print(f"IDs of matching data vectors : {sample_vector_IDs}")
#                 print(f"IDs of matching label vectors: {target_vector_IDs}")
                
                # 'sample_vector_IDs' and 'target_vector_IDs' are IDs of the corresponding peaks. We wish to iterate
                # over 'sample_bragg_peaks' and 'target_bragg_peaks', extracting the relevant Bragg peaks AGAIN IN
                # THE PROPER ORDER, so that returned peaks still have the corresponding order.
                sample_vectors_list  = []
                target_vectors_list = []
                for sample_vector_ID, target_vector_ID in zip(sample_vector_IDs, target_vector_IDs):
                    sample_vector = sample_bragg_peaks[sample_bragg_peaks['ID'] == sample_vector_ID ][0] # !!! '[0]' extracts
                    target_vector = target_bragg_peaks[target_bragg_peaks['ID'] == target_vector_ID][0]  # vector from array.
                    sample_vectors_list.append(sample_vector)
                    target_vectors_list.append(target_vector)
                    
                # Turn the lists into structured arrays before returning.
#                 print(f"\nData vectors list: {sample_vectors_list}.")
                dtype = sample_bragg_peaks.dtype
                sample_vectors_arr  = np.array(sample_vectors_list, dtype=dtype)
#                 print(f"\nData vectors arr: {sample_vectors_arr}.")
                target_vectors_arr = np.array(target_vectors_list, dtype=dtype)
                
                return TRUE_match_verdict, sample_vectors_arr, target_vectors_arr
    
    # If we haven't 'return'-ed by now, TRUE_match_verdict must be false.
    return False, None, None

#################################################

def generate_phase_map(detected_peaks, vacuum_peaks, dataset_4d, target_datapoints,
                       target_bragg_peaks, eps_ratio, eps_angle, eps_length, use_almost = False):
    """
    Runs `check_match` on each real-space position's diffraction pattern to determine if it contains the target
    crystal phase. This gives a 2D true/false map of which positions contain the phase.
    INPUTS
    detected_peaks: (m×n) array where each element is a (qx_list, qy_list) tuple of reciprocal-space peak
                    positions for the corresponding real-space position in the (m×n×ky×kx) 4DSTEM dataset.
    vacuum_peaks: (m×n) array where each element is an array([qx, qy]) giving the vacuum (G=0)
                  peak position for the corresponding real-space position in the 4DSTEM dataset.
    dataset_4d: The 4DSTEM dataset.
    target_datapoints: List of (ratio, angle, length) datapoints corresponding to a crystal phase of interest.
    target_bragg_peaks: length-3 structured NumPy array with fields "ID" and "value", where "value" is a
                        length-2 array [qx, qy] giving Bragg peak positions for the target crystal phase.
    eps_ratio: Ratio tolerance.
    eps_angle: Angle tolerance.
    eps_length: Length tolerance.
    use_almost: A boolean to indicate if we should condition upon length tolerance.
    OUTPUTS
    match_array: (mxn) array with value 1 wherever there is a match, 0 or -5 otherwise.
    darkfield_arr: (mxn) array of darkfield intensities wherever there's a match, -5 elsewhere.
    veci_param_arr: (mxn) array of lattice parameter values. -5 if there's no match.
    """
    
    # Initialize arrays for plotting match info.
    shape_i, shape_j = detected_peaks.shape
    match_array      = np.ones((shape_i, shape_j)) * -5
    
    # Initialize arrays for storing lattice parameters.
    vec0_param_arr     = np.ones((shape_i, shape_j)) * -5
    vec1_param_arr     = np.ones((shape_i, shape_j)) * -5
    vec2_param_arr     = np.ones((shape_i, shape_j)) * -5
    
    # Initialize the darkfield array.
    darkfield_arr      = np.ones((shape_i, shape_j)) * -5
    
    # Iterate over each diffraction pattern.
    for i in range(shape_i):
        for j in range(shape_j):
            print(f"check_match... is on: {(i, j)}")
            
            peaks_ij = detected_peaks[i, j]
            vacuum_point = vacuum_peaks[i, j]
            
            # `peaks_ij` should be a tuple. If it isn't, then no peaks were detected; move on to next pattern.
            if isinstance(peaks_ij, tuple):
                sample_datapoints, sample_bragg_peaks = calculate_datapoints(peaks_ij)

                # Check match and collect crystal basis vectors (if any).
                (match_verdict, sample_vectors_arr,
                 target_vectors_arr)               = check_match(sample_datapoints, sample_bragg_peaks, target_datapoints,
                                                            target_bragg_peaks, eps_ratio, eps_angle, eps_length, use_almost)
                # Save match verdict to array.
                match_array[i, j] = match_verdict

                # If there is a match, store the 'sample' vector lattice parameters.
                if match_verdict:
                    vec0_param_arr[i, j] = np.linalg.norm( sample_vectors_arr[0]['value'] )
                    vec1_param_arr[i, j] = np.linalg.norm( sample_vectors_arr[1]['value'] )
                    vec2_param_arr[i, j] = np.linalg.norm( sample_vectors_arr[2]['value'] )

                    # Calculate the darkfield image intensity for the present diffraction pattern.
                    total_I, _  = make_darkfield(dataset_4d[i, j], sample_vectors_arr,
                                                 vacuum_point, radius=6, include_vacuum=False)
                    darkfield_arr[i, j] = total_I
                
    return match_array, darkfield_arr, vec0_param_arr, vec1_param_arr, vec2_param_arr


# Interfaces for analysis

In [ ]:
## Functions for making virtual detector masks.
def virtual_detector_mask(pattern, row, col, radius):
    """
    Creates a virtual detector with 'radius' at given row-col position of the pattern. Returns the mask
    needed to find darkfield intensity.
    """
    shape = pattern.shape
    y, x = np.meshgrid(np.arange(shape[1]), np.arange(shape[0]))
    mask = ( (x - row)**2 + (y - col)**2 )  <= radius**2
    return mask

#################################################

def make_darkfield(pattern, data_vectors_arr, vacuum_position, radius, include_vacuum=True):
    
    big_mask = np.zeros(pattern.shape)
    
    # Iterate over the various vectors
    for data_vector_single in data_vectors_arr:
        data_vector = data_vector_single['value']
        
        # We now want to "place" multiple virtual detectors around the vacuum position.
        for mult in np.arange(-2, 3, 1).astype(int):
            if mult == 0: # Don't want multiple detectors at vacuum point.
                continue
#             print(f"Mult : {mult}")
            ind = mult * data_vector + vacuum_position
            mask = virtual_detector_mask(pattern, ind[0], ind[1], radius)
            
            # Let's add the mask onto 'big_mask'.
            big_mask = np.logical_or(big_mask, mask)
            
    # Let's also get one detector on vacuum point.
    if include_vacuum:
        mask = virtual_detector_mask(pattern, vacuum_position[0], vacuum_position[1], radius)
        big_mask = np.logical_or(big_mask, mask)
            
    # 'big_mask' should now be a boolean array indexing all pixels for virtual detection. I need to
    # make sure no two virtual detectors overlap- otherwise, it will be non-boolean.
    unique_entries = np.unique(big_mask)
    if not (0 in unique_entries and 1 in unique_entries and len(unique_entries) == 2):
        print("Error! Virtual detector mask is flawed.")
        print(unique_entries)
        print(1/0)
            
    total_intensity = np.sum((pattern[big_mask]).astype(float))
    return total_intensity, big_mask

In [ ]:
import matplotlib.patches as patches
%matplotlib notebook

# Algorithm for rotating 'data' and 'label' Bragg vectors into alignment, and plotting them.
def rot_mat(theta): # Counterclockwise rotation. 'theta' given in radians.
    return np.array([ [np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)] ])

#################################################

def get_angle(v1, v2):
    cos_theta = np.dot(v1, v2) / ( np.linalg.norm(v1) * np.linalg.norm(v2) )
    # Avoid errors due to floating point arithmetic.
    cos_theta = 1.0 if cos_theta >= 1.0 else cos_theta
    return np.arccos(cos_theta)

#################################################

def rotator(data_vectors, label_vectors):
    # Extract the first pair of vectors, for comparison.
    data_vector  = data_vectors[0]['value']
    label_vector = label_vectors[0]['value']
    
    # Find the angle between.
    angle = get_angle(data_vector, label_vector)
        
    # Remember, we are trying to rotate the label vector onto the data one.
    sign = np.sign( np.cross(label_vector, data_vector) )
    
    # Rotate each of the label vectors.
    rotated_label_vectors = np.zeros(3, dtype=label_vectors.dtype)
    
    for i, label_vector_inarray in enumerate(label_vectors): # 'inarray' meaning structured.
        label_vector = label_vector_inarray['value']
        current_id   = label_vector_inarray['ID'] # Let's preserve ID.
        new_vector   = np.matmul( rot_mat(sign * angle), label_vector )
        rotated_label_vectors[i]['value'] = new_vector
        rotated_label_vectors[i]['ID']    = current_id
        
    return rotated_label_vectors

#################################################

def make_rectangle(x, y, width, height, color='red'):
    """
    INPUTS
    x, y: The indices of the center of the rectangle.
    width: Horizontal dimension.
    height: Vertical dimension.
    
    NOTE: x and width refer to horizontal of the plot, y and height to the vertical.
    """
    half_width, half_height = width / 2, height / 2
    anchor_x = x - half_width
    anchor_y = y - half_height
    
    rect = patches.Rectangle((anchor_x, anchor_y), width, height,
                            edgecolor=color, facecolor='none', linewidth=1.5)
    
    return rect

#################################################

# datapoints_gui will go here.

In [ ]:
# # Plotting the eps_X boxes for dset1 and dset2 together on 1 graph; will we see an overlap?
# def compare_dsets(list_of_datapoints, datapoint_labels, eps_ratio, eps_angle):
#     """
#     Plots the (ratio, angle) datapoints for different datasets. Each element in the list corresponds
#     to datapoints from one dataset (i.e. one crystal structure).
#     """
#     fig, ax = plt.subplots()
#     ax.grid(visible=True)
    
#     cmap = plt.colormaps['tab20']
#     colors = [cmap(i) for i in range(len(list_of_datapoints))]

#     for (color, datapoint, label) in zip(colors, list_of_datapoints, datapoint_labels):
#         ratios = datapoint['ratio']
#         angles = datapoint['angle']
#         ax.scatter(ratios, angles, facecolor=color, alpha=0.5, label=label)
#         # Put the eps_X boxes.
#         for ratio, angle in zip(ratios, angles):
#             rect = make_rectangle(ratio, angle, 2*eps_ratio, 2*eps_angle, color=color)
#             ax.add_patch(rect)
            
#     ax.set_xlabel("Ratio")
#     ax.set_ylabel("Angle")
#     ax.legend()
#     plt.show()

In [ ]:
global_checker = 0
def datapoints_gui(vacuum_point, all_detected_peaks, selected_peaks, diffraction_pattern,
                      target_datapoints, target_bragg_peaks, eps_ratio, eps_angle, eps_length):
    global global_checker
    """
    Calculates Datapoints for the given diffraction pattern. Plots Datapoints on one panel and Bragg peaks on another.
    Clicking on a datapoint highlights the 2 Bragg peaks which produced it.
    
    INPUTS
    vacuum_point: Array([qx, qy]) of f the vacuum (G=0) peak's coordinates.
    all_detected_peaks: Tuple (qx_list, qy_list) of all peaks detected in the current diffraction pattern.
    selected_peaks: Tuple (qx_list, qy_list) of detected peaks that were selected by one of two algorithms.
    diffraction_pattern: Self explanatory.
    target_datapoints: Three Datapoints of the crystal phase we are looking for in the diffraction pattern.
    target_bragg_peaks: Three peaks of the same crystal phase.
    eps_ratio, eps_angle, eps_length: Error allowance in Datapoints components.
    dp_vac: DEPRECATED. We already have info of where vacuum point is in every diffraction pattern (from 1st algorithm).
    use_dbscanner: DEPRECATED. Instead should tell which algorithm we want to use (like in other gui).
    
    NOTE: "Peaks" refers to detected Bragg peaks, while "vectors" refers specifically to the 3 vectors
    which identify a crystal pattern.
    """
    
    ################################## First set of calculations. ##################################
        
    # Calculate the Datapoints. We will also attach IDs to the peaks, in array `sample_bragg_peaks`.
    sample_datapoints, sample_bragg_peaks = calculate_datapoints(selected_peaks)

    # Check the match states.
    (match_verdict, 
     sample_vectors_arr, target_vectors_arr) = check_match(sample_datapoints, sample_bragg_peaks, target_datapoints,
                                      target_bragg_peaks, eps_ratio, eps_angle, eps_length, use_almost = False)
    
    # Add vacuum point to align peaks to the diffraction pattern.
    # TODO: The following line is quite messy. Should we change (qx_list, qy_list) -> (qx_arr, qy_arr) everywhere?
    # In fact, why not just use n x 2 arrays? `vacuum_points_arr` is already a `dtype='object'` array where each
    # element is itself a Numpy array, so "arrays of arrays" shouldn't behave strangely.
    selected_peaks = ( np.array(selected_peaks[0]) + vacuum_point[0], np.array(selected_peaks[1]) + vacuum_point[1] )
    sample_bragg_peaks['value'] += vacuum_point
    global_checker = selected_peaks

    # If there was a match, prepare a virtual detector mask and align the sample and target basis vectors.
    if match_verdict:
        # Prepare the virtual detector mask.
        _, detector_mask = make_darkfield(diffraction_pattern, sample_vectors_arr,
                                          vacuum_point, radius=6, include_vacuum=False)

        # Rotate the target vectors into alignment with the diffraction pattern's basis vectors.
        target_vectors_arr = rotator(sample_vectors_arr, target_vectors_arr)
    
    ################################# Generation of initial plots. #################################

    # Set up the figure and subplots. 'ax_datapoints' and 'ax_diffraction' are the main important ones;
    # 'ax_log' is added so that statements can be "logged" to it. Regular "print" statements don't work
    # because the matplotlib's interactive loop overrides jupyter's own one, so strings are printed
    # into a void. Logging can be removed once errors are debugged.
#     fig, (ax_datapoints, ax_diffraction) = plt.subplots(1, 2, figsize=(10, 4))
    fig, ((ax_datapoints, ax_diffraction), (ax_log, _)) = plt.subplots(2, 2, figsize=(10, 8))
    ax_log.axis('off')
    _.axis('off')
    
    # BELOW: Code for logging print statements to 'ax_log'.
# Text object for logging
    log_text = ax_log.text(0, 1, "", va='top', fontsize=10, family='monospace')
    log_lines = []

    def log(msg):
        nonlocal log_lines
        if not isinstance(msg, str):
            msg = f"{msg}"
        log_lines.append(msg)
        log_text.set_text("\n".join(log_lines[-12:]))  # keep last few lines.
        fig.canvas.draw_idle()
    
    ######### Plotting on the panel with diffraction pattern. #########
    # Plot diffraction pattern & its detected peaks.
    im_pattern = ax_diffraction.imshow(np.log(diffraction_pattern), cmap='grey')
    ax_diffraction.set_title("Diffraction pattern")
    
    # Plot all detected Bragg peaks, aligned with the pattern. Remember to swap x and y coordinates.
    ax_diffraction.scatter(all_detected_peaks[1], all_detected_peaks[0], s=200,
                           facecolors='none', edgecolors='blue', label='all peaks')

    # Repeat with peaks selected by one of the two algorithms.
    ax_diffraction.scatter(selected_peaks[1], selected_peaks[0], s=200,
                           facecolors='none', edgecolors='green', label='alg 1/2')
    
    # Plot vacuum point in purple.
    ax_diffraction.scatter(vacuum_point[1], vacuum_point[0], s=200,
                           facecolors='none', edgecolors='purple', label='vacuum')
    
    # If there was a match, plot basis vectors (both sample and target) and the virtual detector mask.
    if match_verdict:
        log("A match occured!")
        # Plot the sample and (rotated) target Bragg vectors. There are 3 vector coordinates in each.
        ax_diffraction.scatter(sample_vectors_arr['value'][:, 1] + vacuum_point[1],
                               sample_vectors_arr['value'][:, 0] + vacuum_point[0],
                               marker='s', alpha=0.5, s=60, facecolors='cyan', edgecolors='none',
                               label = r'sample $\vec{v}$')

        ax_diffraction.scatter(target_vectors_arr['value'][:, 1] + vacuum_point[1],
                               target_vectors_arr['value'][:, 0] + vacuum_point[0],
                               alpha=0.5, s=60, facecolors='orange', edgecolors='none',
                              label = r'target $\vec{v}$')
        
        # Plot the virtual detector. Make it invisible.
        virtual_detector_artist = ax_diffraction.imshow(detector_mask, cmap='viridis', alpha=0.4)
        virtual_detector_artist.set_visible(False)
        
    ################ Plotting on the Datapoints panel. ################
    # Start by placing the 'target' points & tolerance boxes.
    ratios = target_datapoints['ratio']
    angles = target_datapoints['angle']
    ax_datapoints.scatter(ratios, angles, facecolor='orange', s=20, alpha=0.5, label='target')
    
    # Put the eps_X boxes.
    for ratio, angle in zip(ratios, angles):
        rect = make_rectangle(ratio, angle, 2*eps_ratio, 2*eps_angle, color='orange')
        ax_datapoints.add_patch(rect)
            
    # Then, plot the 'sample' datapoints.
    ax_datapoints.scatter(sample_datapoints['ratio'], sample_datapoints['angle'],
                          facecolor='blue', s=20, alpha=0.5, label='sample')
    ax_datapoints.set_xlim(left=0.9) # Ratios are greater than or equal to 1.0
    
    # Put match verdict in title.
    ax_datapoints.set_title(f"Match: {match_verdict}")
    
    log("Initial plotting of Datapoints panel successful.")
    ################################## Interactive plot elements. ##################################
        
    # Make Artist objects which will get updated by user interactions. Make them invisible until needed.
    selected_datapoint_artist = ax_datapoints.scatter(0, 0, s=200, facecolors='none',
                                                      edgecolors='red', label='highlighted')
    selected_datapoint_artist.set_visible(False)
    selected_peaks_artist = ax_diffraction.scatter(0,0, s=200, facecolors='none',
                                                    edgecolors='red', label='highlighted')
    selected_peaks_artist.set_visible(False)
    
    # Set up the initial legend.
    make_legend(ax_datapoints)
    make_legend(ax_diffraction)
    
    def onpress(event):
        nonlocal match_verdict, virtual_detector_artist
        log("`onpress` function is activated.")
        # If there's a match and 'v' is pressed, reverse the visibility of the virtual detector.
        if event.key =='v' and match_verdict:
            virtual_detector_artist.set_visible(not virtual_detector_artist.get_visible())
            fig.canvas.draw_idle()
        
    def onclick(event):
        nonlocal selected_datapoint_artist, selected_peaks_artist
        log("`onclick` function is activated.")
        
        # Do nothing if a tool on the Toolbar is selected (e.g. zoom).
        toolbar = event.canvas.toolbar
        if toolbar is not None and toolbar.mode != '':
            return

        # Do nothing if the mouseclick isn't in one of the two interesting plot panels.
        if event.inaxes != ax_datapoints and event.inaxes != ax_diffraction:
            return
            
        # Depending on which panel was clicked, we want to do different things.
        if event.inaxes == ax_datapoints:
            log("Selected a Datapoint.")
            ## We want to highlight the sample Datapoint nearest to the mouse click, and also
            ## the two corresponding Bragg peaks which formed it.
            
            # Highlight the nearest sample Datapoint.
            idx = highlight_nearest_point(ax_datapoints, event, sample_datapoints['ratio'],
                                          sample_datapoints['angle'], selected_datapoint_artist)
            log("Successfully highlighted clicked Datapoint.")

            # Next, find the pair of Bragg peaks which produced this point. Highlight them.
            peaks_to_plot = [] # Stores selected peaks' coordinates for plotting.
            pair_ids = sample_datapoints['pair_ID'][idx]
            for i, ID in enumerate(pair_ids):
                # Look for this ID in the detected Bragg peaks.
                peak_idx = (ID == sample_bragg_peaks['ID']) # Boolean indices.
                peak_value = sample_bragg_peaks['value'][peak_idx][0] # Gives Array([a, b]).
                log(f"peak_value: {peak_value}")

                # Store the peak's coordinates for plotting. Reverse x and y axes.
                peaks_to_plot.append( peak_value[::-1] )
                log("Successfully stored position of 1 peak.")

            # Highlight the two peaks.
            selected_peaks_artist.set_offsets(peaks_to_plot)
            selected_peaks_artist.set_visible(True)
            log("Successfully highlighted the 2 peaks responsible for Datapoint.")
        else:
            log("Selected a Bragg peak.")
            ## We want to highlight the sample Bragg peak nearest to the mouse click, and also highlight
            ## all Datapoints which it takes part in producing.
            
            # Highlight the nearest Bragg peak. TODO: the x and y axis swapping stuff is extra confusing
            # here; I should consider renaming to "rows" and "cols" globally, to aleviate mental suffering.
            idx = highlight_nearest_point(ax_diffraction, event, sample_bragg_peaks['value'][:, 1],
                                          sample_bragg_peaks['value'][:, 0], selected_peaks_artist)
            log(f"Highlighted Bragg peak with index = {idx}. Value: {sample_bragg_peaks['value'][idx]}")
        
            # Extract the ID of this peak, and find all Datapoints which have that ID in their `pair_ID`.
            ID = sample_bragg_peaks['ID'][idx]
            datapoints_to_plot = [] # Stores selected Datapoints' coordinates for plotting.
            for current_datapoint in sample_datapoints:
                if ID in current_datapoint['pair_ID']:
                    datapoints_to_plot.append( (current_datapoint['ratio'], current_datapoint['angle']) )
            
            # Highlight these Datapoints.
            selected_datapoint_artist.set_offsets(datapoints_to_plot)
            selected_datapoint_artist.set_visible(True)
            log("Successfully highlighted all the Datapoints from current peak.")
        
        
        
        # Update the figure (including both subplots).
#         ax_datapoints.legend()
#         ax_diffraction.legend()
        make_legend(ax_datapoints)
        make_legend(ax_diffraction)
        fig.canvas.draw_idle()
        log("Successfully updated the figure after highlights.")

    log("Following this log, button/key press events will be connected.")
    fig.canvas.mpl_connect("button_press_event", onclick)
    fig.canvas.mpl_connect("key_press_event", onpress)
#     ax_datapoints.legend()
    make_legend(ax_datapoints)
    ax_datapoints.set_xlabel("Ratio")
    ax_datapoints.set_ylabel("Angle")
    plt.show()
    
    return sample_bragg_peaks, 0

def make_legend(ax):
    """Used to make a legend for a figure, or update existing one upon user interaction."""
    leg = ax.get_legend()
    if leg:
        leg.remove()
    ax.legend(fontsize=8, markerscale=0.5,
              labelspacing=0.2, handletextpad=0.3,
              borderpad=0.3, handlelength=1)
    

In [ ]:
global_checker = 0
def select_target_vectors(diffraction_pattern, selected_peaks, vacuum_point, qx_list, qy_list):
    # Plot the diffraction pattern.
    fig, ax = plt.subplots(figsize=(4,4))
    ax.imshow(np.log(diffraction_pattern), cmap='grey')
    ax.set_title("Select 3 peaks.")
    
    # Plot the selected peaks on top of the diffraction pattern.
    aligned_peaks_x = selected_peaks[1] + vacuum_point[1]
    aligned_peaks_y = selected_peaks[0] + vacuum_point[0]
    ax.scatter(aligned_peaks_x, aligned_peaks_y, s=200, facecolors='none', edgecolors='blue', label='peaks')
    
    # Initialize Artist object which highlights the peaks that the user will select.
    artist = ax.scatter(0, 0, s=200, facecolors='none', edgecolors='green', label='user selection')
    artist.set_visible(False)
    
    # Draw the canvas.
    ax.legend()
    fig.canvas.draw_idle()
    
    def onclick(event):
        nonlocal qx_list, qy_list
        
        # Do nothing if toolbar is in use.
        toolbar = event.canvas.toolbar
        if toolbar is not None and toolbar.mode != '':
            return
        
        # Do nothing if click is outside figure panel.
        if event.inaxes != ax:
            return
        
        # Do nothing if 3 peaks are already selected.
        if len(qx_list) >= 3:
            return
        
        # Find the nearest peak. Save it's coordinates. TODO: Fix horrible x/y thing.
        idx = highlight_nearest_point(ax, event, aligned_peaks_x, aligned_peaks_y, artist, modify_artist=False)
        qx_list.append(selected_peaks[0][idx])
        qy_list.append(selected_peaks[1][idx])
        
        # Highlight the peaks selected thus far.
        artist.set_offsets([(y + vacuum_point[1], x + vacuum_point[0]) for x, y in zip(qx_list, qy_list)])
        artist.set_visible(True)
        
        # Update canvas.
        ax.legend()
        fig.canvas.draw_idle()
        
    fig.canvas.mpl_connect('button_press_event', onclick)
    plt.tight_layout()
    plt.show()

In [ ]:
def highlight_nearest_point(ax, event, x_data, y_data, artist, modify_artist=True):
    """Finds which point is nearest to the mouse click, highlights it and returns its index.
    INPUTS
    ax : The axis of the scatterplot.
    event: The mouse click event, which contains position.
    x_data, y_data: 1D containers containing the x- and y-axis coordinates of points. NOTE: These are the actual
                    coordinates on the plot; in previous functions, `x` refers to rows of 2D matrices which are
                    actually plotted on the y-axis (which isn't the case here).
    artist: The Artist object which we update to cause the highlighting effect.
    OUTPUTS
    idx: The index of the highlighted datapoint. Used for later analysis.
    """
    
    # We need the extents of the axes in the current plot panel. To get Euclidean distances
    # between mouse and datapoint/peak positions, we must normalize by this range.
    x_min, x_max = ax.get_xlim()
    y_min, y_max = ax.get_ylim()
    x_range = x_max - x_min
    y_range = y_max - y_min

    # Compute nearest point.
    distances = np.hypot((x_data - event.xdata) / x_range,
                         (y_data - event.ydata) / y_range)
    idx = np.argmin(distances)

    # Highlight clicked datapoint.
    if modify_artist:
        artist.set_offsets([x_data[idx], y_data[idx]])
        artist.set_visible(True)

    return idx

In [ ]:
def gui(data4d, brightfield, detected_peaks_4d=None, selected_peaks_4d=None, vacuum_point_4d=None):
    m, n = brightfield.shape   
    
    # Plot the brightfield image. It will never be changed/updated.
    fig, (ax_img, ax_dp) = plt.subplots(1, 2, figsize=(10, 4))
    im = ax_img.imshow(brightfield, cmap='grey')
    ax_img.set_title("Brightfield (click a pixel)")
    dp_plot = ax_dp.imshow(np.zeros(data4d.shape[2:]), cmap='gray')
    ax_dp.set_title("Click a pixel → diffraction here")
    
    # Initialize `Artist` objects which will get updated interactively.
    selected_pixel_artist = ax_img.scatter(0, 0, s=1, facecolors='red')
    selected_pixel_artist.set_visible(False)
    
    detected_peaks_artist = ax_dp.scatter(0,0, s=200, facecolors='none', edgecolors='blue', label='all peaks')
    detected_peaks_artist.set_visible(False)

    selected_peaks_artist = ax_dp.scatter(0,0, s=200, facecolors='none', edgecolors='green', label='selected')
    selected_peaks_artist.set_visible(False)

    vacuum_point_artist   = ax_dp.scatter(0, 0, s=200, facecolors='none', edgecolors='purple', label='vacuum')
    vacuum_point_artist.set_visible(False)
        
    # Initialize the plot.
    ax_dp.legend()
    fig.canvas.draw_idle()
    
    
    def onclick(event):
        if event.inaxes != ax_img:
            return
        mpl_x = int(round(event.xdata))
        mpl_y = int(round(event.ydata))
        
        # Clear the detected/selected/vacuum peaks plotted from previous click.
        [_.set_visible(False) for _ in (selected_peaks_artist, detected_peaks_artist, vacuum_point_artist)]
        
        # Remember, the 2D 'imshow' plot has rows on y-axis and columns on x-axis. So, we
        # must index as data[y, x].
        np_x = mpl_y
        np_y = mpl_x
        if 0 <= np_x < m and 0 <= np_y < n:
            # Plot a dot on the selected pixel of the darkfield/boolean map.
            selected_pixel_artist.set_offsets([np_y, np_x])
            selected_pixel_artist.set_visible(True)
            
            # Extract and plot the diffraction pattern selected from dataset.
            dp = data4d[np_x, np_y]
            dp_log = np.log(dp)
            dp_plot.set_data(dp_log)
            dp_plot.set_clim(vmin=dp_log.min(), vmax=dp_log.max())
            ax_dp.set_title(f"Diffraction at index [{np_x}, {np_y}]")
            
            # Plot additional detected/selected peaks, if given. TODO: Axis reversal is again messy. Needs removal.
            if detected_peaks_4d is not None and isinstance(detected_peaks_4d[np_x, np_y], tuple):
                detected_peaks = detected_peaks_4d[np_x, np_y]
                detected_peaks_artist.set_offsets(np.column_stack([detected_peaks[1], detected_peaks[0]]))
                detected_peaks_artist.set_visible(True)
            if vacuum_point_4d is not None:
                vacuum_point = vacuum_point_4d[np_x, np_y]
                vacuum_point_artist.set_offsets([vacuum_point[1], vacuum_point[0]])
                vacuum_point_artist.set_visible(True)
            if (vacuum_point_4d is not None and selected_peaks_4d is not None
                and isinstance(selected_peaks_4d[np_x, np_y], tuple)):
                selected_peaks = selected_peaks_4d[np_x, np_y]
                selected_peaks_artist.set_offsets(np.column_stack([selected_peaks[1]+vacuum_point[1],
                                                                   selected_peaks[0]+vacuum_point[0]]))
                selected_peaks_artist.set_visible(True)
            
            # Update the canvas.
            ax_dp.legend()
            fig.canvas.draw_idle()
    
    fig.canvas.mpl_connect('button_press_event', onclick)
    plt.tight_layout()
    plt.show()


# <center>Beginning of Analysis</center>

# Defining "target" Datapoints

We will start by using `gui` to click through various pixels of the brightfield image, until we find clear diffraction patterns. We need one pattern each for WZ, ZB and Pb.

In [ ]:
brightfield = np.load("brightfield.npy", allow_pickle=True)
gui(pattern_data, brightfield, detected_peaks_4d=detected_peaks_arr,
    selected_peaks_4d=alg_2_peaks_arr, vacuum_point_4d=vacuum_peaks_arr)

## Target Datapoints for Pb

Index [11, 15] was the Lead (Pb) pattern.

In [ ]:
idx_x, idx_y = 11, 15
qx_list, qy_list = [], []
select_target_vectors(pattern_data[idx_x, idx_y], alg_2_peaks_arr[idx_x, idx_y],
                      vacuum_peaks_arr[idx_x, idx_y], qx_list, qy_list)

In [ ]:
# Use this to calculate Datapoints.
datapoints_Pb, bragg_peaks_Pb = calculate_datapoints( (qx_list, qy_list) )
print(datapoints_Pb)
print(bragg_peaks_Pb)

## Target Datapoints for ZB

Index [61, 73] was a zincblende (ZB) diffraction pattern.

In [ ]:
idx_x, idx_y = 61, 73
qx_list, qy_list = [], []
select_target_vectors(pattern_data[idx_x, idx_y], alg_2_peaks_arr[idx_x, idx_y],
                      vacuum_peaks_arr[idx_x, idx_y], qx_list, qy_list)

In [ ]:
# Use this to calculate Datapoints.
datapoints_ZB, bragg_peaks_ZB = calculate_datapoints( (qx_list, qy_list) )
print(datapoints_ZB)
print(bragg_peaks_ZB)

## Target Datapoints for WZ

Index [51, 28] was a wurtzite (WZ) pattern.

In [ ]:
idx_x, idx_y = 51, 28
qx_list, qy_list = [], []
select_target_vectors(pattern_data[idx_x, idx_y], alg_2_peaks_arr[idx_x, idx_y],
                      vacuum_peaks_arr[idx_x, idx_y], qx_list, qy_list)

In [ ]:
# Use this to calculate Datapoints.
datapoints_WZ, bragg_peaks_WZ = calculate_datapoints( (qx_list, qy_list) )
print(datapoints_WZ)
print(bragg_peaks_WZ)

## Defining match tolerances

We don't expect target and sample Datapoints to match exactly; we may allow for some variation in the components of a Datapoint (ratio, angle, length). Below we define the relevant hyperparameters.

In [ ]:
eps_ratio  = 0.12
eps_angle  = 6
eps_length = 2

# Generating phase maps

Now that we have target Datapoints for ZB, WZ and Pb, we can generate phase maps for them. A phase map is a 2D map where each pixel has value 1 if the corresponding position has the target crystal phase, 0 otherwise.

(Actually, pixels have value -5 if algorithm 2 is used and no peaks are detected in a given pattern. That is equivalent to saying there's no crystal.)

## Phase map for Pb

In [ ]:
(Pb_phase_map, _, _, _, _) = generate_phase_map(alg_2_peaks_arr, vacuum_peaks_arr,
                                                pattern_data, datapoints_Pb, bragg_peaks_Pb,
                                                eps_ratio, eps_angle, eps_length, use_almost = False)

In [ ]:
plt.imshow(Pb_phase_map==1, cmap='grey')
plt.show()

## Phase map for ZB

Remember that ZB and Pb have the same reciprocal lattice, with differing lattice parameters. We can directly check this by setting `use_almost=False` in the following code, which will then highlight pixels where zincblende OR lead are present. I won't do this here.

In [ ]:
(ZB_phase_map, _, _, _, _) = generate_phase_map(alg_2_peaks_arr, vacuum_peaks_arr,
                                                pattern_data, datapoints_ZB, bragg_peaks_ZB,
                                                eps_ratio, eps_angle, eps_length, use_almost = False)

In [ ]:
plt.imshow(ZB_phase_map==1, cmap='grey')
plt.show()

## Phase map for WZ

In [ ]:
(WZ_phase_map, _, _, _, _) = generate_phase_map(alg_2_peaks_arr, vacuum_peaks_arr,
                                                pattern_data, datapoints_WZ, bragg_peaks_WZ,
                                                eps_ratio, eps_angle, eps_length, use_almost = False)

In [ ]:
plt.imshow(WZ_phase_map==1, cmap='grey')
plt.show()

## Combined phase map

We can combine the 3 phase maps above, giving each a different color, to observe the prevalence of the 3 desired crystal phases throughout the nanowire.

In [ ]:
import matplotlib.patches as mpatches

im1 = plt.imshow(Pb_phase_map==1, cmap='Blues', alpha=1)
im2 = plt.imshow(WZ_phase_map==1, cmap='Greens', alpha=0.5)
im3 = plt.imshow(ZB_phase_map==1, cmap='Reds', alpha=0.3)

# Create legend patches
pb_patch = mpatches.Patch(color='blue', label='Pb')
wz_patch = mpatches.Patch(color='green', label='WZ')
zb_patch = mpatches.Patch(color='red', label='ZB')

plt.legend(handles=[pb_patch, wz_patch, zb_patch])

plt.xlabel("$r_x$")
plt.ylabel("$r_y$")
plt.show()

In [ ]:
# np.save('Pb_map.npy', Pb_phase_map==1, allow_pickle=True) # We saved the other 2 before.